# Kaggle - Brain Tumor MRI Dataset

You can find the dataset and some informations about on the [Kaggle page](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).

For details on steps below, please see documentation in the *docs* directory.

## Kaggle notebook setup
### Installations

In [1]:
!pip install mlflow --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 73.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.3/788.3 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires g

### Data location

In [2]:
import os
print(os.listdir('/kaggle/input/'))
print(os.listdir('/kaggle/input/radimagenet-densenet121-notop'))
print(os.listdir('/kaggle/input/brain-tumor-mri-preprocessed'))
print(os.listdir('/kaggle/input/brain-tumor-mri-preprocessed/processed'))

['radimagenet-densenet121-notop', 'brain-tumor-mri-preprocessed']
['RadImageNet-DenseNet121_notop.h5']
['processed']
['Validation', 'Training', 'Testing', '.gitkeep']


## General

In [3]:
import mlflow
from mlflow.tracking import MlflowClient
mlflow.set_tracking_uri("https://preachingly-nonabjuratory-marget.ngrok-free.dev")
mlflow.set_experiment("Brain_Tumor_Training")

<Experiment: artifact_location='mlflow-artifacts:/830660600119881173', creation_time=1769099937797, experiment_id='830660600119881173', last_update_time=1769099937797, lifecycle_stage='active', name='Brain_Tumor_Training', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [4]:
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
import math

2026-01-26 15:23:54.652877: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769441034.841042      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769441034.898734      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769441035.383884      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769441035.383930      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769441035.383932      55 computation_placer.cc:177] computation placer alr

In [5]:
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
tf.config.list_physical_devices('GPU')

Num GPUs Available: 1


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [6]:
# path management
PROJECT_ROOT = '/kaggle/input'
PREP_DIR = PROJECT_ROOT + '/brain-tumor-mri-preprocessed/processed'
ARTEFACTS_DIR = PROJECT_ROOT + '/radimagenet-densenet121-notop'

CLASSES = ["notumor", "glioma", "meningioma", "pituitary"]

# parameters
IMG_SIZE = 260
SEED = 42

PROJECT_NAME = "BrainTumorMRI"
MODEL_TYPE = "DenseNet121"
TWO_HEAD = True
MODEL_NAME = f"{PROJECT_NAME}_{MODEL_TYPE}_{TWO_HEAD*"2Head"}"
FREEZE_BACKBONE = True
MASK_TUMOR_TYPE_LOSS = True
BATCH_SIZE = 32

## Modeling

### Backbone

In [7]:
# 1. Create DenseNet121 WITHOUT weights
backbone = DenseNet121(
    include_top=False,
    weights=None,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 2. Load RadImageNet weights
backbone.load_weights(ARTEFACTS_DIR + "/RadImageNet-DenseNet121_notop.h5")

# 3. Freeze the backbone for firsts training
backbone.trainable = not FREEZE_BACKBONE

print("✅ RadImageNet DenseNet121 loaded successfully")

I0000 00:00:1769441048.440276      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


✅ RadImageNet DenseNet121 loaded successfully


In [8]:
#backbone.summary()

In [9]:
w = backbone.weights[0].numpy()
print("Mean:", np.mean(w), "Std:", np.std(w))

Mean: -0.0028284893 Std: 0.10494729


Model seems to be correctly loaded.

### Model definition

In [ ]:
model_data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation((-0.09,0.09), seed=SEED),
    layers.RandomZoom((-0.03,0.03),(-0.03,0.03), seed=SEED),
    layers.RandomTranslation((-0.01,0.01),(-0.01,0.01), seed=SEED),
    layers.RandomContrast(0.02, seed=SEED),
    # To use parcimoniously -> off in fine-tuning
    layers.RandomBrightness(0.005, seed=SEED),
    layers.GaussianNoise(0.005, seed=SEED)
], name='data_augmentation_part')

In [10]:
model_shared_part = keras.Sequential([
    #Data augmentation
    model_data_augmentation,
    # Base
    backbone,
    # Head
    layers.GlobalAveragePooling2D(), # to flatten backbone output but with moderate position importance and more stable for MRI
    layers.Dense(512, use_bias=False), # 512 because it half of the backbone output (1024)
    layers.BatchNormalization(), # to normalize weight before heads
    layers.Activation('relu'),
    layers.Dropout(0.4) # to reduce over-fitting risks
], name='shared_part')

In [11]:
model_head1 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(1,activation='sigmoid')
], name='tumor_presence')

In [12]:
model_head2 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(4,activation='softmax')
], name='tumor_type')

In [13]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = model_shared_part(inputs)
output_presence = model_head1(x)
output_type = model_head2(x)

model = keras.Model(
    inputs=inputs,
    outputs={
        "tumor_presence": output_presence,
        "tumor_type": output_type
    },
    name='densenet_two_head'
)

In [14]:
loss_presence = keras.losses.BinaryFocalCrossentropy(
    gamma=2.0,
    alpha=0.25 # to favorize tumor detection, but taking account that tumors are 75% of data
)

In [15]:
def masked_sparse_cce(y_true, y_pred):
    tumor_present = tf.cast(y_true != 0, tf.float32)
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    loss = loss * tumor_present
    return tf.reduce_sum(loss) / (tf.reduce_sum(tumor_present) + 1e-6)

In [16]:
model.compile(
    optimizer=keras.optimizers.Adam(), # change learning_rate for 1e-4 in fine-tuning steps
    loss={
        "tumor_presence": loss_presence,
        "tumor_type": masked_sparse_cce,
    },

    metrics={
        "tumor_presence": [
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.AUC(name="auc")
        ],
        "tumor_type": ["accuracy"],
    }
)

In [17]:
#model.summary()

## Streaming Training

In [18]:
def parse_tfrecord(example_proto):
    """
    Parse a single TFRecord example and convert grayscale → RGB.
    """
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }

    example = tf.io.parse_single_example(example_proto, feature_description)

    # Deserialize image
    image = tf.io.parse_tensor(example["image"], out_type=tf.float32)

    # Shape after loading: (260, 260, 3)
    image.set_shape((260, 260, 3))

    label = tf.cast(example["label"], tf.int32)

    return image, label



def load_tfrecord_dataset(tfrecord_dir, shuffle=False, batch_size=1, repeat=False):
    """
    Load a TFRecord dataset from a directory.

    Args:
        tfrecord_dir (str or Path): Folder containing .tfrecord files
        shuffle (bool): Whether to shuffle files and samples
        batch_size (int): Batch size (can stay 1)
        repeat (bool): Repeat dataset indefinitely (for training)

    Returns:
        tf.data.Dataset
    """
    tfrecord_files = tf.io.gfile.glob(
        str(tfrecord_dir) + "/*.tfrecord"
    )

    ds = tf.data.TFRecordDataset(
        tfrecord_files,
        num_parallel_reads=tf.data.AUTOTUNE
    )

    ds = ds.map(
        parse_tfrecord,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        ds = ds.shuffle(buffer_size=512)

    if repeat:
        ds = ds.repeat()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


In [19]:
TRAIN_DIR = PREP_DIR + "/Training"
VAL_DIR   = PREP_DIR + "/Validation"
TEST_DIR  = PREP_DIR + "/Testing"

train_ds = load_tfrecord_dataset(
    TRAIN_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)

val_ds = load_tfrecord_dataset(
    VAL_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)

"""
test_ds = load_tfrecord_dataset(
    TEST_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)
"""

'\ntest_ds = load_tfrecord_dataset(\n    TEST_DIR,\n    shuffle=False,\n    batch_size=BATCH_SIZE,\n    repeat=False\n).prefetch(tf.data.AUTOTUNE)\n'

In [20]:
def split_labels(image, label):
    """
    Create labels for a 2-head model.
    """
    tumor_present = tf.cast(label != 0, tf.float32)
    tumor_type = tf.cast(label, tf.int32)

    return image, {
        "tumor_presence": tumor_present,
        "tumor_type": tumor_type
    }


In [21]:
train_ds = train_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)

In [22]:
#for x, y in train_ds.take(1):
#    print("Image:")
#    print(x.dtype, x.shape)
#    print("\nLabels:")
#    for k, v in y.items():
#        print(k, v.dtype, v.shape)

In [23]:
def count_tfrecord_batches(directory_path, batch_size):
    """
    Count number of batches for a TFRecord dataset.

    Assumes:
    - 1 sample per TFRecord
    - batch_size is variable

    Returns:
    - number of batches = ceil(num_samples / batch_size)
    """
    directory_path = Path(directory_path)
    n_samples = len(list(directory_path.glob("*.tfrecord")))
    return math.ceil(n_samples / batch_size)


In [24]:
print(train_ds.element_spec)

(TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name=None), {'tumor_presence': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'tumor_type': TensorSpec(shape=(None,), dtype=tf.int32, name=None)})


In [29]:
reduce_lr = ReduceLROnPlateau(
    monitor="val_tumor_presence_recall",
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_tumor_presence_recall",
    mode="max",
    min_delta=0.0001,
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

In [30]:
#print("RUN_NAME:", RUN_NAME)
#print("TRAIN steps:", count_tfrecord_batches(TRAIN_DIR, BATCH_SIZE))
#print("VAL steps:", count_tfrecord_batches(VAL_DIR, BATCH_SIZE))
#print("train_ds element_spec:", train_ds.element_spec)

In [ ]:
RUN_NAME = (
    f"{MODEL_TYPE}"
    f"freeze={FREEZE_BACKBONE}_"
    f"mask={MASK_TUMOR_TYPE_LOSS}_"
    f"{datetime.now().strftime('%Y%m%d-%H%M')}"
)

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.tensorflow.autolog(registered_model_name=MODEL_NAME)

    mlflow.log_params({
        "project": PROJECT_NAME,
        "model_type": MODEL_TYPE,
        "pretrained_weights": "RadImageNet",
        "backbone_frozen": FREEZE_BACKBONE,
        "head_1": "tumor_presence_binary",
        "head_2": "tumor_type_softmax",
        "mask_tumor_type_loss": MASK_TUMOR_TYPE_LOSS,
        "label_0_means": "no_tumor",
    })

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=50,
        #steps_per_epoch=count_tfrecord_batches(TRAIN_DIR, BATCH_SIZE),
        #validation_steps=count_tfrecord_batches(VAL_DIR, BATCH_SIZE),
        callbacks=[reduce_lr, early_stopping],
        verbose=1,
    )

    client = MlflowClient()
    latest_version = client.get_latest_versions(MODEL_NAME)[-1].version

    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest_version,
        stage="Staging"
    )


2026/01/26 15:51:22 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.
2026/01/26 15:51:24 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.


Epoch 1/50
    143/Unknown 11s 72ms/step - loss: 0.4407 - tumor_presence_accuracy: 0.9298 - tumor_presence_auc: 0.9694 - tumor_presence_loss: 0.0532 - tumor_presence_precision: 0.9468 - tumor_presence_recall: 0.9571 - tumor_type_accuracy: 0.6176 - tumor_type_loss: 0.3875

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


143/143 ━━━━━━━━━━━━━━━━━━━━ 31s 212ms/step - loss: 0.4406 - tumor_presence_accuracy: 0.9299 - tumor_presence_auc: 0.9694 - tumor_presence_loss: 0.0531 - tumor_presence_precision: 0.9468 - tumor_presence_recall: 0.9571 - tumor_type_accuracy: 0.6176 - tumor_type_loss: 0.3874 - val_loss: 1.3167 - val_tumor_presence_accuracy: 0.9475 - val_tumor_presence_auc: 0.9784 - val_tumor_presence_loss: 0.0424 - val_tumor_presence_precision: 0.9526 - val_tumor_presence_recall: 0.9757 - val_tumor_type_accuracy: 0.3683 - val_tumor_type_loss: 1.2729 - learning_rate: 2.5000e-04
Epoch 2/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 0.3897 - tumor_presence_accuracy: 0.9486 - tumor_presence_auc: 0.9841 - tumor_presence_loss: 0.0400 - tumor_presence_precision: 0.9624 - tumor_presence_recall: 0.9669 - tumor_type_accuracy: 0.6301 - tumor_type_loss: 0.3498

143/143 ━━━━━━━━━━━━━━━━━━━━ 27s 190ms/step - loss: 0.3897 - tumor_presence_accuracy: 0.9486 - tumor_presence_auc: 0.9841 - tumor_presence_loss: 0.0400 - tumor_presence_precision: 0.9624 - tumor_presence_recall: 0.9669 - tumor_type_accuracy: 0.6301 - tumor_type_loss: 0.3497 - val_loss: 0.6510 - val_tumor_presence_accuracy: 0.9466 - val_tumor_presence_auc: 0.9734 - val_tumor_presence_loss: 0.0450 - val_tumor_presence_precision: 0.9569 - val_tumor_presence_recall: 0.9697 - val_tumor_type_accuracy: 0.5276 - val_tumor_type_loss: 0.6056 - learning_rate: 2.5000e-04
Epoch 3/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 13s 89ms/step - loss: 0.3722 - tumor_presence_accuracy: 0.9510 - tumor_presence_auc: 0.9851 - tumor_presence_loss: 0.0378 - tumor_presence_precision: 0.9560 - tumor_presence_recall: 0.9770 - tumor_type_accuracy: 0.6172 - tumor_type_loss: 0.3344 - val_loss: 0.7893 - val_tumor_presence_accuracy: 0.9668 - val_tumor_presence_auc: 0.9838 - val_tumor_presence_loss: 0.0423 - val_tumor_presence_prec

143/143 ━━━━━━━━━━━━━━━━━━━━ 26s 177ms/step - loss: 0.3511 - tumor_presence_accuracy: 0.9567 - tumor_presence_auc: 0.9843 - tumor_presence_loss: 0.0362 - tumor_presence_precision: 0.9648 - tumor_presence_recall: 0.9760 - tumor_type_accuracy: 0.6324 - tumor_type_loss: 0.3149 - val_loss: 0.5328 - val_tumor_presence_accuracy: 0.9396 - val_tumor_presence_auc: 0.9909 - val_tumor_presence_loss: 0.0368 - val_tumor_presence_precision: 0.9285 - val_tumor_presence_recall: 0.9927 - val_tumor_type_accuracy: 0.5669 - val_tumor_type_loss: 0.4956 - learning_rate: 2.5000e-04
Epoch 6/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 13s 87ms/step - loss: 0.3359 - tumor_presence_accuracy: 0.9594 - tumor_presence_auc: 0.9881 - tumor_presence_loss: 0.0325 - tumor_presence_precision: 0.9724 - tumor_presence_recall: 0.9718 - tumor_type_accuracy: 0.6424 - tumor_type_loss: 0.3034 - val_loss: 0.8279 - val_tumor_presence_accuracy: 0.8985 - val_tumor_presence_auc: 0.9879 - val_tumor_presence_loss: 0.0602 - val_tumor_presence_prec

143/143 ━━━━━━━━━━━━━━━━━━━━ 23s 160ms/step - loss: 0.3119 - tumor_presence_accuracy: 0.9623 - tumor_presence_auc: 0.9905 - tumor_presence_loss: 0.0294 - tumor_presence_precision: 0.9718 - tumor_presence_recall: 0.9765 - tumor_type_accuracy: 0.6435 - tumor_type_loss: 0.2825 - val_loss: 0.3511 - val_tumor_presence_accuracy: 0.9589 - val_tumor_presence_auc: 0.9948 - val_tumor_presence_loss: 0.0257 - val_tumor_presence_precision: 0.9512 - val_tumor_presence_recall: 0.9939 - val_tumor_type_accuracy: 0.6387 - val_tumor_type_loss: 0.3271 - learning_rate: 1.2500e-04
Epoch 11/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - loss: 0.2913 - tumor_presence_accuracy: 0.9679 - tumor_presence_auc: 0.9941 - tumor_presence_loss: 0.0236 - tumor_presence_precision: 0.9792 - tumor_presence_recall: 0.9766 - tumor_type_accuracy: 0.6526 - tumor_type_loss: 0.2677

143/143 ━━━━━━━━━━━━━━━━━━━━ 27s 184ms/step - loss: 0.2913 - tumor_presence_accuracy: 0.9679 - tumor_presence_auc: 0.9941 - tumor_presence_loss: 0.0236 - tumor_presence_precision: 0.9792 - tumor_presence_recall: 0.9766 - tumor_type_accuracy: 0.6525 - tumor_type_loss: 0.2677 - val_loss: 0.3093 - val_tumor_presence_accuracy: 0.9755 - val_tumor_presence_auc: 0.9954 - val_tumor_presence_loss: 0.0197 - val_tumor_presence_precision: 0.9795 - val_tumor_presence_recall: 0.9867 - val_tumor_type_accuracy: 0.6457 - val_tumor_type_loss: 0.2902 - learning_rate: 1.2500e-04
Epoch 12/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 13s 88ms/step - loss: 0.2696 - tumor_presence_accuracy: 0.9658 - tumor_presence_auc: 0.9942 - tumor_presence_loss: 0.0234 - tumor_presence_precision: 0.9777 - tumor_presence_recall: 0.9754 - tumor_type_accuracy: 0.6575 - tumor_type_loss: 0.2462 - val_loss: 0.4232 - val_tumor_presence_accuracy: 0.9755 - val_tumor_presence_auc: 0.9956 - val_tumor_presence_loss: 0.0199 - val_tumor_presence_pre

2026/01/26 15:55:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/26 15:56:00 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmplqf1tujs/model, flavor: tensorflow). Fall back to return ['tensorflow==2.19.0', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 


In [ ]:
Warning : do not forget :
- stratégie de fine-tuning du backbone
- BatchNormalization precaution in fine-tuning
- Data aug precaution in fine-tuning
- maximize reccal (y_pred > 0.3 → tumeur)